# Model A — Anemia Hemoglobin (Hb) Regression

This notebook implements and evaluates the **Model A (ResNet-50)** regressor predicting numerical laboratory hemoglobin (Hb in g/dL) from non-invasive ocular images.

### Clinical Constraints Enforced:
1. **Never fabricate ground truth**: Strict pairing with laboratory Hb spreadsheets (`India.xlsx`, `Italy.xlsx`).
2. **Strict Patient Leakage Prevention**: All multi-view images belonging to the same subject reside strictly in the same split (Train, Val, or Test).
3. **Color-space preservation**: No aggressive color jitter that would distort conjunctival erythema/pallor.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np

# Add workspace root
WORKSPACE_ROOT = os.path.abspath('..')
if WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, WORKSPACE_ROOT)

from src.preprocessing.dataset_anemia import build_anemia_dataloaders
from src.models.anemia_model import AnemiaRegressor, load_anemia_checkpoint
from src.evaluation.evaluate import compute_regression_metrics

print('Torch version:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Processed Manifests and DataLoaders

In [ ]:
manifest_path = os.path.join(WORKSPACE_ROOT, 'data', 'manifests', 'anemia_manifest.csv')
df = pd.read_csv(manifest_path)
print(f'Total records in manifest: {len(df)}')
print('Split distribution by subject:')
print(df.groupby('split')['subject_id'].nunique())

loaders = build_anemia_dataloaders(
    manifest_csv=manifest_path,
    workspace_root=WORKSPACE_ROOT,
    batch_size=16,
    use_processed_dir=os.path.join(WORKSPACE_ROOT, 'data', 'processed', 'anemia')
)

## 2. Load Trained ResNet50 Regressor & Evaluate

In [ ]:
checkpoint_path = os.path.join(WORKSPACE_ROOT, 'saved_models', 'anemia', 'best_model.pth')
if os.path.exists(checkpoint_path):
    model = load_anemia_checkpoint(checkpoint_path)
    print('Loaded checkpoint from:', checkpoint_path)
else:
    print('Checkpoint not found. Run scripts/train_anemia.py first.')

In [ ]:
# Evaluate on test loader
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in loaders['test']:
        images = batch['image']
        targets = batch['hb']
        preds = model(images)
        all_preds.extend(preds.numpy())
        all_targets.extend(targets.numpy())

metrics = compute_regression_metrics(np.array(all_targets), np.array(all_preds))
print('=== Test Evaluation Metrics ===')
print(f'MAE:  {metrics["mae_g_dl"]} g/dL')
print(f'RMSE: {metrics["rmse_g_dl"]} g/dL')
print(f'R²:   {metrics["r2"]}')
print(f'Pearson r: {metrics["pearson_r"]}')